# 10 — Backpropagation: one number at a time

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/010-backpropagation.ipynb)

Training means finding which way to move each number to make the loss smaller. Backpropagation is the chain rule applied carefully, and here you do it on a single weight and bias, check it against a nudge, and then break the sign.

## 1. The numbers

The model is one weight and one bias, the input is 2, and the target is 10. The loss is the squared gap between the guess and the target, so a wrong guess is punished more than proportionally.

In [ ]:
import numpy as np                                    # the array library, preinstalled on Colab

x = 2.0                                               # one input
w = 3.0                                               # the weight we are learning
b = -1.0                                              # the bias we are learning
y = 10.0                                              # the target the model should have produced

pred = w * x + b                                      # forward pass: the model's guess
err = pred - y                                        # how far the guess is from the target
loss = err ** 2                                       # squared error: the one number to minimise
print("prediction =", pred)                           # 5.0
print("error =", err)                                 # -5.0, the guess is too low
print("loss =", loss)                                 # 25.0

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — loads NumPy, the array library Colab already has.</li><li><code>x = 2.0</code> — the single input.</li><li><code>w = 3.0</code> — the weight being learned.</li><li><code>b = -1.0</code> — the bias being learned.</li><li><code>y = 10.0</code> — the answer the model should produce.</li><li><code>pred = w * x + b</code> — the forward pass.</li><li><code>err = pred - y</code> — the residual, signed: negative means too low.</li><li><code>loss = err ** 2</code> — squaring removes the sign and punishes large errors.</li><li><code>print("prediction", ...)</code> — the guess.</li><li><code>print("error", ...)</code> — how far off, with a sign.</li><li><code>print("loss", ...)</code> — the number training will push down.</li></ul>

</details>

## 2. Step by step: the backward pass

Backpropagation walks the graph backwards, multiplying one local slope at a time. The nudge check then measures the same slope without any calculus, which is how you catch a mistake.

In [ ]:
dL_dpred = 2 * err                                     # slope of the square: 2 times the error
dL_dw = dL_dpred * x                                  # chain rule: times d(pred)/dw, which is x
dL_db = dL_dpred * 1.0                                # chain rule: d(pred)/db is 1
print(f"by hand:  dL/dw = {dL_dw:+.2f}   dL/db = {dL_db:+.2f}")   # -20 and -10

eps = 1e-4                                            # a tiny nudge
L_plus = ((w + eps) * x + b - y) ** 2                 # loss with the weight nudged up
L_minus = ((w - eps) * x + b - y) ** 2                # loss with the weight nudged down
numeric = (L_plus - L_minus) / (2 * eps)              # slope measured by the two nudges
print(f"nudged:   dL/dw = {numeric:+.2f}")            # -20, same as the hand version
print(f"difference = {abs(numeric - dL_dw):.2e}")     # ~1e-11: rounding only

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>dL_dpred = 2 * err</code> — derivative of a square, at the current error.</li><li><code>dL_dw = dL_dpred * x</code> — chain rule through the multiply.</li><li><code>dL_db = dL_dpred * 1.0</code> — chain rule through the add; the slope is 1.</li><li><code>print(f"by hand: ...")</code> — the two slopes, from calculus only.</li><li><code>eps = 1e-4</code> — the size of the nudge.</li><li><code>L_plus</code> — the loss one nudge above the current weight.</li><li><code>L_minus</code> — the loss one nudge below it.</li><li><code>numeric = (L_plus - L_minus) / (2 * eps)</code> — rise over run, symmetric.</li><li><code>print(f"nudged: ...")</code> — the same slope, measured without calculus.</li><li><code>abs(numeric - dL_dw)</code> — the two answers should agree to about ten digits.</li></ul>

</details>

## 3. Plot: the loss, with the gradient as an arrow

The loss as a function of one weight is a parabola, and its lowest point is the weight that predicts the target exactly. The gradient is the slope under your feet; the arrow shows the step you take against it.

In [ ]:
import matplotlib.pyplot as plt                        # plotting library, preinstalled on Colab

w_grid = np.linspace(-1, 12, 200)                     # weights from -1 to 12
loss_grid = (w_grid * x + b - y) ** 2                 # the loss at every one of them

fig, ax = plt.subplots(figsize=(6, 3))                # one axis, wide and short
ax.plot(w_grid, loss_grid, color="tab:blue", label="loss(w)")   # the parabola
ax.scatter([w], [loss], color="tab:red", zorder=3, label="where we are")   # current weight
step_w = w - 0.15 * dL_dw                             # move against the slope
step_loss = (step_w * x + b - y) ** 2                 # the loss at the new weight
ax.annotate("", xy=(step_w, step_loss), xytext=(w, loss), arrowprops=dict(arrowstyle="->", color="tab:red", lw=2))   # the descent arrow
ax.axvline((y - b) / x, color="grey", linestyle="--", label="best w")   # the bottom of the parabola
ax.set_xlabel("weight w")                             # x-axis label
ax.set_ylabel("loss")                                 # y-axis label
ax.set_title("the gradient points down the parabola")  # the takeaway
ax.legend()                                           # name the marks
plt.show()                                            # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import matplotlib.pyplot as plt</code> — Colab already ships the plotting library.</li><li><code>w_grid = np.linspace(-1, 12, 200)</code> — a range of candidate weights.</li><li><code>loss_grid = ...</code> — the loss for each candidate, one array.</li><li><code>plt.subplots(figsize=(6, 3))</code> — a figure and its single axis.</li><li><code>ax.plot(...)</code> — the parabola itself.</li><li><code>ax.scatter([w], [loss])</code> — marks the current (weight, loss) pair.</li><li><code>step_w = w - 0.15 * dL_dw</code> — a small step against the gradient.</li><li><code>step_loss = ...</code> — the loss at the end of that step.</li><li><code>ax.annotate("", xy=..., ...)</code> — an arrow from here to there; empty text.</li><li><code>ax.axvline((y - b) / x, ...)</code> — the weight that makes the error zero.</li><li><code>ax.set_xlabel("weight w")</code> — names the horizontal axis.</li><li><code>ax.set_ylabel("loss")</code> — names the vertical axis.</li><li><code>ax.set_title(...)</code> — the takeaway.</li><li><code>ax.legend()</code> — matches each mark to its label.</li><li><code>plt.show()</code> — draws the figure.</li></ul>

</details>

## 4. Interactive: put the weight anywhere

Drag the weight across the parabola. The readout gives the loss and the slope, and you can watch the slope shrink to zero as the dot reaches the bottom.

In [ ]:
from ipywidgets import interact                        # sliders that rerun a function

def show(w_value=3.0):                                # w_value comes from the slider
    p = w_value * x + b                               # forward pass at this weight
    e = p - y                                         # residual
    g = 2 * e * x                                     # d(loss)/dw at this weight
    plt.figure(figsize=(6, 3))                        # fresh figure on every drag
    plt.plot(w_grid, loss_grid, color="tab:blue")     # the same fixed parabola
    plt.scatter([w_value], [e ** 2], color="tab:red", zorder=3)   # current point
    plt.axvline((y - b) / x, color="grey", linestyle="--")   # the minimum
    plt.xlabel("weight w")                            # x-axis label
    plt.ylabel("loss")                                # y-axis label
    plt.title(f"w = {w_value:.2f} -> loss = {e ** 2:.2f}, slope = {g:+.2f}")   # live readout
    plt.show()                                        # draw it

interact(show, w_value=(-1.0, 12.0, 0.5))             # attach the slider

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>from ipywidgets import interact</code> — Colab ships the widgets library.</li><li><code>def show(w_value=3.0)</code> — the function that redraws one frame.</li><li><code>p = ...</code> — a forward pass at the slider's weight.</li><li><code>e = p - y</code> — the residual, signed.</li><li><code>g = 2 * e * x</code> — the gradient, computed in one line.</li><li><code>plt.figure(...)</code> — a new canvas for each frame.</li><li><code>plt.plot(...)</code> — the same fixed parabola.</li><li><code>plt.scatter(...)</code> — the current point on it.</li><li><code>plt.axvline(...)</code> — the bottom of the parabola.</li><li><code>plt.xlabel("weight w")</code> — names the horizontal axis.</li><li><code>plt.ylabel("loss")</code> — names the vertical axis.</li><li><code>plt.title(f"...")</code> — loss and slope, both live.</li><li><code>plt.show()</code> — draws the frame.</li><li><code>interact(show, w_value=(...))</code> — the slider, its range and its step.</li></ul>

</details>

## 5. When it breaks: the wrong sign

Gradient descent says subtract the gradient. Add it instead and every step climbs the parabola, which turns a small error into a huge one. The two curves below differ only in that one sign.

In [ ]:
lr = 0.02                                             # step size, shared by both runs
w_ok, w_bad = 3.0, 3.0                                # the same starting weight
ok_hist, bad_hist = [], []                            # one loss per step

for step in range(30):                                # thirty steps each
    g_ok = 2 * (w_ok * x + b - y) * x                 # gradient for the correct run
    w_ok = w_ok - lr * g_ok                           # subtract: downhill
    ok_hist.append((w_ok * x + b - y) ** 2)           # record the loss
    g_bad = 2 * (w_bad * x + b - y) * x               # gradient for the broken run
    w_bad = w_bad + lr * g_bad                        # ADD: uphill, the classic sign slip
    bad_hist.append((w_bad * x + b - y) ** 2)         # record the loss

print(f"correct sign: w = {w_ok:.3f}  loss = {ok_hist[-1]:.4f}")   # settles near 5.5
print(f"wrong sign:   w = {w_bad:.1f}  loss = {bad_hist[-1]:.2e}")  # runs away
print(f"loss grew {bad_hist[-1] / bad_hist[0]:.0f}x in 30 steps")  # growth, not shrinkage

plt.figure(figsize=(6, 3))                            # new figure
plt.plot(ok_hist, color="tab:green", label="w = w - lr * slope")   # falls
plt.plot(bad_hist, color="tab:red", label="w = w + lr * slope")    # rises
plt.yscale("log")                                     # the two scales differ wildly
plt.xlabel("step")                                    # x-axis label
plt.ylabel("loss (log scale)")                        # y-axis label
plt.title("a sign flip turns descent into divergence")  # the failure
plt.legend()                                          # name the curves
plt.show()                                            # render

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>lr = 0.02</code> — the step size used by both runs.</li><li><code>w_ok, w_bad = 3.0, 3.0</code> — identical starting points, so only the sign differs.</li><li><code>ok_hist, bad_hist = [], []</code> — one loss list per run.</li><li><code>for step in range(30)</code> — thirty steps for each rule.</li><li><code>g_ok = ...</code> — the gradient for the correct run.</li><li><code>w_ok = w_ok - lr * g_ok</code> — the correct update.</li><li><code>ok_hist.append(...)</code> — record its loss.</li><li><code>g_bad = ...</code> — the same gradient formula for the broken run.</li><li><code>w_bad = w_bad + lr * g_bad</code> — the broken update; <code>+</code> instead of <code>-</code>.</li><li><code>bad_hist.append(...)</code> — record its loss.</li><li><code>print(f"correct sign ...")</code> — where the good run ended up.</li><li><code>print(f"wrong sign ...")</code> — how far the bad run flew.</li><li><code>print(f"loss grew ...x")</code> — the ratio makes the divergence explicit.</li><li><code>plt.figure(...)</code> — a new canvas.</li><li><code>plt.plot(ok_hist, ...)</code> — the green curve, falling.</li><li><code>plt.plot(bad_hist, ...)</code> — the red curve, rising.</li><li><code>plt.yscale("log")</code> — a log axis, because the red curve leaves the chart.</li><li><code>plt.xlabel("step")</code> — names the horizontal axis.</li><li><code>plt.ylabel(...)</code> — notes that the axis is logged.</li><li><code>plt.title(...)</code> — the failure in one line.</li><li><code>plt.legend()</code> — names which line is which update rule.</li><li><code>plt.show()</code> — draws the figure.</li></ul>

</details>

## Your turn

In section 4, find the weight where the slope reads 0.00 and compare it with `(y - b) / x`. Then set `lr = 0.05` in section 5 and predict how many steps the correct run needs to get within 0.01 of the bottom.